In [0]:
%pip install python-dotenv --quiet
dbutils.library.restartPython()

In [0]:
# Conectando via abfss a Tabela food_lotes_producao.csv
import os
from dotenv import load_dotenv

load_dotenv("/Workspace/Users/ik.kukoo@gmail.com/.env", override=True)

STORAGE_ACCOUNT = "internshipdatalake"
CONTAINER       = "raw"
CONTAINER_PATH  = "batch-data"
TABLE           = "food_lotes_producao.csv"

adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

PATH_PRODUCAO = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CONTAINER_PATH}/{TABLE}"

print("✅ Config OK →", PATH_PRODUCAO)

In [0]:
# Leitura da Tabela food_avaliacoes_produto.csv
df = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PATH_PRODUCAO)
)

print(f"✅ {df.count()} linhas | {len(df.columns)} colunas")
df.printSchema()

In [0]:
# Amostra da Tabela
df.show(10, truncate=False)

In [0]:
# Estatísticas Descritivas da Tabela
df.describe().show(truncate=False)

In [0]:
# Análise de Nulos da Tabela
from pyspark.sql.functions import col, sum as spark_sum

nulls = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
print("🔍 Nulos por coluna:")
nulls.show(truncate=False)

In [0]:
# Distribuição por status
from pyspark.sql.functions import count, round as spark_round

total = df.count()

df.groupBy("status").agg(
    count("*").alias("quantidade")
).withColumn(
    "percentual",
    spark_round((col("quantidade") / total) * 100, 2)
).orderBy("quantidade", ascending=False).show(truncate=False)

In [0]:
# Lotes vencidos por SKU
df.filter(col("status") == "Vencido") \
  .groupBy("sku").count() \
  .orderBy("count", ascending=False) \
  .show(20, truncate=False)

In [0]:
# Temperatura de armazenamento
df.groupBy("temperatura_armazenamento_ideal").count() \
  .orderBy("count", ascending=False) \
  .show(truncate=False)

In [0]:
# Quantidade produzida por fornecedor
from pyspark.sql.functions import avg, sum as spark_sum, max as spark_max

df.groupBy("id_fornecedor").agg(
    count("*").alias("total_lotes"),
    spark_sum("quantidade_produzida").alias("total_produzido"),
    avg("quantidade_produzida").alias("media_por_lote"),
    spark_max("quantidade_produzida").alias("maior_lote")
).orderBy("total_produzido", ascending=False).show(20, truncate=False)

In [0]:
# Validade média por temperatura de armazenamento
from pyspark.sql.functions import datediff, avg

df.withColumn(
    "dias_validade",
    datediff(col("dt_validade"), col("dt_fabricacao"))
).groupBy("temperatura_armazenamento_ideal").agg(
    avg("dias_validade").alias("media_dias_validade")
).orderBy("media_dias_validade", ascending=False).show(truncate=False)